# European Green Crab Detection

This notebook trains and evaluates a YOLO model using the local `All-Data-1` dataset.

## 1. Install Dependencies

Install the Python packages required for training, validation, and visualization.

In [ ]:
%pip install -q ultralytics opencv-python-headless pillow matplotlib pyyaml

## 2. Locate the Local Dataset

Use the `All-Data-1` folder next to this notebook and load its dataset configuration.

In [ ]:
from pathlib import Path
import yaml

# Use the dataset stored beside this notebook.
workspace_dir = Path.cwd()
dataset_root = workspace_dir / "All-Data-1"
data_yaml_path = dataset_root / "data.yaml"

if not data_yaml_path.is_file():
    raise FileNotFoundError(
        f"Could not find the local dataset configuration at {data_yaml_path}. "
        "Run this notebook from the workspace containing All-Data-1."
    )

with data_yaml_path.open("r", encoding="utf-8") as file:
    data_config = yaml.safe_load(file) or {}

data_config["path"] = str(dataset_root)

with data_yaml_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(data_config, file, sort_keys=False)

print("Using local dataset:", dataset_root)
print("Dataset configuration:", data_yaml_path)
print(data_config)

## 3. Prepare Dataset Configuration

Ensure YOLO uses the absolute local dataset path.

In [ ]:
from pathlib import Path
import yaml

data_yaml_path = Path.cwd() / "All-Data-1" / "data.yaml"
dataset_root = data_yaml_path.parent

if not data_yaml_path.is_file():
    raise FileNotFoundError(f"Could not find local dataset configuration: {data_yaml_path}")

with data_yaml_path.open("r", encoding="utf-8") as file:
    data_config = yaml.safe_load(file) or {}

data_config["path"] = str(dataset_root)

with data_yaml_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(data_config, file, sort_keys=False)

print(f"Using local data.yaml: {data_yaml_path}")
print(f"Dataset root: {dataset_root}")

## 4. Train the YOLO Model

Train on the local dataset with GPU support when available and CPU fallback otherwise.

In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("yolo11s.pt")
device = 0 if torch.cuda.is_available() else "cpu"

results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=16 if device != "cpu" else 4,
    device=device,
    workers=2,
    patience=20,
)

print("Training completed! Check runs/detect/train/weights/best.pt for outputs.")

## 5. Run a Local Test Prediction

Run inference on the first image found in the local test split and display the annotated result.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
from ultralytics import YOLO

model_path = Path("runs/detect/train/weights/best.pt")
model = YOLO(str(model_path))

image_source = Path.cwd() / "crab.jpg"
if not image_source.is_file():
    raise FileNotFoundError(f"Could not find the test image: {image_source}")

results = model.predict(source=str(image_source), conf=0.40, save=True)

save_dir = Path(results[0].save_dir)
output_path = save_dir / image_source.name

if output_path.exists():
    display(Image(filename=str(output_path)))
else:
    print(f"Could not find output image at: {output_path}")

## 6. Evaluate Model Accuracy

Validate the trained weights on the validation split and print precision and recall.

In [ ]:
from ultralytics import YOLO

model_path = "runs/detect/train/weights/best.pt"
model = YOLO(model_path)

metrics = model.val(data=str(data_yaml_path), split="val")

print("\n--- Accuracy Summary ---")
print(f"Precision:    {metrics.results_dict['metrics/precision(B)']:.4f}")
print(f"Recall:       {metrics.results_dict['metrics/recall(B)']:.4f}")